# FlyCeNN vs Transformer — real FlyWire topology probe

This Colab tests whether a sparse recurrent/nonlinear model constrained by **real FlyWire v783 biological connectivity** can perform the sequence-model role of a small Transformer. It compares a parameter-matched Transformer, biological FlyCeNN, and a rewired FlyCeNN control on identical byte-level language-model batches. Metrics include cross-entropy, bits/byte, perplexity, throughput, peak VRAM, and a topology-gain report.

Biological graph source: Zenodo `10.5281/zenodo.21549559`, derived from FlyWire v783. The connectome supplies the wiring prior; recurrence, nonlinear dynamics, trainable edge gains, input mapping, and readout are artificial learned components.


In [ ]:
import base64
exec(base64.b64decode('CmltcG9ydCBvcywgbWF0aCwgdGltZSwganNvbiwgcmFuZG9tLCBwYXRobGliLCBzdWJwcm9jZXNzLCBzeXMsIHJlcXVlc3RzCmltcG9ydCBudW1weSBhcyBucCwgcGFuZGFzIGFzIHBkLCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdApmcm9tIHRxZG0uYXV0byBpbXBvcnQgdHFkbQoKU0VFRD00MjsgTk9ERVM9MjU2OyBNQVhfRURHRVM9MTUzNjsgU0VRPTk2OyBCQVRDSD0xNjsgU1RFUFM9MTYwOyBFTUI9MTI4CnJhbmRvbS5zZWVkKFNFRUQpOyBucC5yYW5kb20uc2VlZChTRUVEKTsgdG9yY2gubWFudWFsX3NlZWQoU0VFRCkKZGV2aWNlPXRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQpwcmludCgiZGV2aWNlOiIsZGV2aWNlLCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSBpZiBkZXZpY2UudHlwZT09ImN1ZGEiIGVsc2UgIiIpCnJlcG89cGF0aGxpYi5QYXRoKCIvY29udGVudC9UaW55Q2VOTi1MTSIpCmlmIHJlcG8uZXhpc3RzKCk6CiAgICBzdWJwcm9jZXNzLnJ1bihbImdpdCIsIi1DIixzdHIocmVwbyksImZldGNoIiwib3JpZ2luIl0sY2hlY2s9VHJ1ZSkKICAgIHN1YnByb2Nlc3MucnVuKFsiZ2l0IiwiLUMiLHN0cihyZXBvKSwicmVzZXQiLCItLWhhcmQiLCJvcmlnaW4vbWFpbiJdLGNoZWNrPVRydWUpCmVsc2U6CiAgICBzdWJwcm9jZXNzLnJ1bihbImdpdCIsImNsb25lIiwiaHR0cHM6Ly9naXRodWIuY29tL3Z0YXZha2tvbGkvVGlueUNlTk4tTE0uZ2l0IixzdHIocmVwbyldLGNoZWNrPVRydWUpCgojIC0tLSByZWFsIEZseVdpcmUgdjc4MyBiaW9sb2dpY2FsIGdyYXBoLCBwcm9jZXNzZWQgZGF0YXNldCBET0kgMTAuNTI4MS96ZW5vZG8uMjE1NDk1NTkKY2FjaGU9cGF0aGxpYi5QYXRoKCIvY29udGVudC9mbHljZW5uX2NhY2hlIik7IGNhY2hlLm1rZGlyKGV4aXN0X29rPVRydWUpCmd6PWNhY2hlLyJjb25uZWN0aW9uc19iaW9sb2dpY2FsLmNzdi5neiIKdXJsPSJodHRwczovL3plbm9kby5vcmcvcmVjb3Jkcy8yMTU0OTU1OS9maWxlcy9jb25uZWN0aW9uc19iaW9sb2dpY2FsLmNzdi5nej9kb3dubG9hZD0xIgppZiBub3QgZ3ouZXhpc3RzKCk6CiAgICB3aXRoIHJlcXVlc3RzLmdldCh1cmwsc3RyZWFtPVRydWUsdGltZW91dD02MCkgYXMgcjoKICAgICAgICByLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgICAgIHdpdGggb3Blbihneiwid2IiKSBhcyBmOgogICAgICAgICAgICBmb3IgYyBpbiB0cWRtKHIuaXRlcl9jb250ZW50KDEwMjQqMTAyNCksZGVzYz0iRmx5V2lyZSBncmFwaCIpOgogICAgICAgICAgICAgICAgaWYgYzogZi53cml0ZShjKQoKY29scz1bInByZV9yb290X2lkIiwicG9zdF9yb290X2lkIiwic3luX2NvdW50Il0KZGVnPXBkLlNlcmllcyhkdHlwZT1ucC5mbG9hdDY0KQpmb3IgY2ggaW4gdHFkbShwZC5yZWFkX2Nzdihneix1c2Vjb2xzPWNvbHMsY29tcHJlc3Npb249Imd6aXAiLGNodW5rc2l6ZT0xXzAwMF8wMDApLGRlc2M9ImRlZ3JlZSBwYXNzIik6CiAgICB3PW5wLmxvZzFwKGNoLnN5bl9jb3VudC50b19udW1weShucC5mbG9hdDY0KSkKICAgIGE9cGQuU2VyaWVzKHcsaW5kZXg9Y2gucHJlX3Jvb3RfaWQudG9fbnVtcHkoKSkuZ3JvdXBieShsZXZlbD0wKS5zdW0oKQogICAgYj1wZC5TZXJpZXModyxpbmRleD1jaC5wb3N0X3Jvb3RfaWQudG9fbnVtcHkoKSkuZ3JvdXBieShsZXZlbD0wKS5zdW0oKQogICAgZGVnPWRlZy5hZGQoYSxmaWxsX3ZhbHVlPTApLmFkZChiLGZpbGxfdmFsdWU9MCkKY2FuZD1zZXQoaW50KGspIGZvciBrIGluIGRlZy5ubGFyZ2VzdChOT0RFUyozKS5pbmRleCkKcGFydHM9W10KZm9yIGNoIGluIHRxZG0ocGQucmVhZF9jc3YoZ3osdXNlY29scz1jb2xzLGNvbXByZXNzaW9uPSJnemlwIixjaHVua3NpemU9MV8wMDBfMDAwKSxkZXNjPSJlZGdlIHBhc3MiKToKICAgIHE9Y2hbY2gucHJlX3Jvb3RfaWQuaXNpbihjYW5kKSZjaC5wb3N0X3Jvb3RfaWQuaXNpbihjYW5kKV0KICAgIGlmIGxlbihxKTogcGFydHMuYXBwZW5kKHEpCmU9cGQuY29uY2F0KHBhcnRzKS5ncm91cGJ5KFsicHJlX3Jvb3RfaWQiLCJwb3N0X3Jvb3RfaWQiXSxhc19pbmRleD1GYWxzZSkuc3luX2NvdW50LnN1bSgpLnNvcnRfdmFsdWVzKCJzeW5fY291bnQiLGFzY2VuZGluZz1GYWxzZSkKbm9kZXM9W107IHNlZW49c2V0KCkKZm9yIHIgaW4gZS5oZWFkKE1BWF9FREdFUyo2KS5pdGVydHVwbGVzKGluZGV4PUZhbHNlKToKICAgIGZvciB4IGluIChpbnQoci5wcmVfcm9vdF9pZCksaW50KHIucG9zdF9yb290X2lkKSk6CiAgICAgICAgaWYgeCBub3QgaW4gc2Vlbjogc2Vlbi5hZGQoeCk7IG5vZGVzLmFwcGVuZCh4KQogICAgICAgIGlmIGxlbihub2Rlcyk+PU5PREVTOiBicmVhawogICAgaWYgbGVuKG5vZGVzKT49Tk9ERVM6IGJyZWFrCnM9c2V0KG5vZGVzKQplPWVbZS5wcmVfcm9vdF9pZC5pc2luKHMpJmUucG9zdF9yb290X2lkLmlzaW4ocyldLmhlYWQoTUFYX0VER0VTKS5jb3B5KCkKYWN0aXZlPXNvcnRlZChzZXQoZS5wcmVfcm9vdF9pZC5hc3R5cGUoaW50KSl8c2V0KGUucG9zdF9yb290X2lkLmFzdHlwZShpbnQpKSk7IG1wPXt4OmkgZm9yIGkseCBpbiBlbnVtZXJhdGUoYWN0aXZlKX0Kc3JjPWUucHJlX3Jvb3RfaWQuYXN0eXBlKGludCkubWFwKG1wKS50b19udW1weShucC5pbnQ2NCk7IGRzdD1lLnBvc3Rfcm9vdF9pZC5hc3R5cGUoaW50KS5tYXAobXApLnRvX251bXB5KG5wLmludDY0KQpyYXc9bnAubG9nMXAoZS5zeW5fY291bnQudG9fbnVtcHkobnAuZmxvYXQzMikpOyBpbmM9bnAuemVyb3MobGVuKGFjdGl2ZSksbnAuZmxvYXQzMik7IG5wLmFkZC5hdChpbmMsZHN0LHJhdyk7IGJhc2U9KHJhdy9ucC5tYXhpbXVtKGluY1tkc3RdLDFlLTYpKS5hc3R5cGUobnAuZmxvYXQzMikKZHN0X3JhbmQ9ZHN0LmNvcHkoKTsgbnAucmFuZG9tLmRlZmF1bHRfcm5nKFNFRUQrMSkuc2h1ZmZsZShkc3RfcmFuZCkKTj1sZW4oYWN0aXZlKTsgcHJpbnQoImdyYXBoIixOLCJub2RlcyIsbGVuKHNyYyksImVkZ2VzIikKCiMgLS0tIGJ5dGUtbGV2ZWwgVGlueSBTaGFrZXNwZWFyZSBjb3JwdXMKdHh0PXJlcXVlc3RzLmdldCgiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL2thcnBhdGh5L2NoYXItcm5uL21hc3Rlci9kYXRhL3RpbnlzaGFrZXNwZWFyZS9pbnB1dC50eHQiLHRpbWVvdXQ9NjApLmNvbnRlbnQKYXJyPW5wLmZyb21idWZmZXIodHh0LGR0eXBlPW5wLnVpbnQ4KS5hc3R5cGUobnAuaW50NjQpCmN1dD1pbnQoLjkqbGVuKGFycikpOyB0cj10b3JjaC5mcm9tX251bXB5KGFycls6Y3V0XSk7IHZhPXRvcmNoLmZyb21fbnVtcHkoYXJyW2N1dDpdKQpvZmY9dG9yY2guYXJhbmdlKFNFUSsxKQpnPXRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKFNFRUQrOSkKc3RhcnRzPXRvcmNoLnJhbmRpbnQoMCxsZW4odHIpLVNFUS0yLChTVEVQUyxCQVRDSCksZ2VuZXJhdG9yPWcpCnZzdGFydHM9dG9yY2gucmFuZGludCgwLGxlbih2YSktU0VRLTIsKDI0LEJBVENIKSxnZW5lcmF0b3I9ZykKZGVmIGJhdGNoKGRhdGEsc3QpOgogICAgej1kYXRhW3N0WzosTm9uZV0rb2ZmW05vbmUsOl1dLmxvbmcoKQogICAgcmV0dXJuIHpbOiw6LTFdLnRvKGRldmljZSksels6LDE6XS50byhkZXZpY2UpCgpjbGFzcyBGbHlMTShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsZHN0Xyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpOyBzZWxmLk49TgogICAgICAgIHNlbGYuZW1iPW5uLkVtYmVkZGluZygyNTYsRU1CKTsgc2VsZi5pbnA9bm4uTGluZWFyKEVNQixOKTsgc2VsZi5vdXQ9bm4uTGluZWFyKE4sRU1CKTsgc2VsZi5ub3JtPW5uLkxheWVyTm9ybShFTUIpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoInNyYyIsdG9yY2gudGVuc29yKHNyYykpOyBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiZHN0Iix0b3JjaC50ZW5zb3IoZHN0XykpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImJ3Iix0b3JjaC50ZW5zb3IoYmFzZSkpCiAgICAgICAgc2VsZi5nYWluPW5uLlBhcmFtZXRlcih0b3JjaC5mdWxsKChsZW4oc3JjKSwpLGZsb2F0KG5wLmFyY3RhbmgoMi8zKSkpKQogICAgICAgIHNlbGYuZGVjYXk9bm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKE4pKTsgc2VsZi5zZWxmcz1ubi5QYXJhbWV0ZXIodG9yY2guZnVsbCgoTiwpLC4yNSkpOyBzZWxmLmdhdGU9bm4uUGFyYW1ldGVyKHRvcmNoLmZ1bGwoKE4sKSwtMS4wKSkKICAgIGRlZiBtaXgoc2VsZixoKToKICAgICAgICBCLFQsTj1oLnNoYXBlOyB4PWgucmVzaGFwZSgtMSxOKS5UCiAgICAgICAgdz1zZWxmLmJ3LnRvKHguZHR5cGUpKigxLjUqdG9yY2gudGFuaChzZWxmLmdhaW4pKS50byh4LmR0eXBlKQogICAgICAgIG1zZz14LmluZGV4X3NlbGVjdCgwLHNlbGYuc3JjKSp3WzosTm9uZV0KICAgICAgICB5PXRvcmNoLnplcm9zX2xpa2UoeCk7IHkuaW5kZXhfYWRkXygwLHNlbGYuZHN0LG1zZykKICAgICAgICByZXR1cm4geS5ULnJlc2hhcGUoQixULE4pCiAgICBkZWYgZm9yd2FyZChzZWxmLHRvayk6CiAgICAgICAgeD1zZWxmLmVtYih0b2spOyB1PXRvcmNoLnRhbmgoc2VsZi5pbnAoeCkpOyBhPSguNTUrLjQ0NSp0b3JjaC5zaWdtb2lkKHNlbGYuZGVjYXkpKS50byh1LmR0eXBlKVtOb25lLDpdCiAgICAgICAgaD10b3JjaC56ZXJvcyh0b2suc2l6ZSgwKSxOLGRldmljZT10b2suZGV2aWNlLGR0eXBlPXUuZHR5cGUpOyBocz1bXQogICAgICAgIGZvciB0IGluIHJhbmdlKHRvay5zaXplKDEpKToKICAgICAgICAgICAgaD1hKmgrKDEtYSkqdVs6LHRdOyBocy5hcHBlbmQoaCkKICAgICAgICB6PXRvcmNoLnN0YWNrKGhzLDEpCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoMik6CiAgICAgICAgICAgIHA9dG9yY2gudGFuaChzZWxmLm1peCh6KStzZWxmLnNlbGZzLnRvKHouZHR5cGUpW05vbmUsTm9uZSw6XSp6Ky4yKnUpCiAgICAgICAgICAgIHo9eit0b3JjaC5zaWdtb2lkKHNlbGYuZ2F0ZSkudG8oei5kdHlwZSlbTm9uZSxOb25lLDpdKnAKICAgICAgICBxPXNlbGYubm9ybShzZWxmLm91dCh6KSkKICAgICAgICByZXR1cm4gRi5saW5lYXIocSxzZWxmLmVtYi53ZWlnaHQpCgpjbGFzcyBQb3Mobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLGQpOiBzdXBlcigpLl9faW5pdF9fKCk7IHNlbGYuZD1kCiAgICBkZWYgZm9yd2FyZChzZWxmLHgpOgogICAgICAgIFQ9eC5zaXplKDEpOyBwPXRvcmNoLmFyYW5nZShULGRldmljZT14LmRldmljZSlbOixOb25lXS5mbG9hdCgpOyBrPXRvcmNoLmFyYW5nZShzZWxmLmQvLzIsZGV2aWNlPXguZGV2aWNlKS5mbG9hdCgpCiAgICAgICAgZj10b3JjaC5leHAoLW1hdGgubG9nKDEwMDAwKSprL21heChzZWxmLmQvLzItMSwxKSlbTm9uZSw6XTsgcGU9dG9yY2guemVyb3MoVCxzZWxmLmQsZGV2aWNlPXguZGV2aWNlLGR0eXBlPXguZHR5cGUpCiAgICAgICAgcGVbOiwwOjoyXT10b3JjaC5zaW4ocCpmKS50byh4LmR0eXBlKTsgcGVbOiwxOjoyXT10b3JjaC5jb3MocCpmKS50byh4LmR0eXBlKTsgcmV0dXJuIHgrcGVbTm9uZV0KY2xhc3MgVHJMTShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsZD04MCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpOyBzZWxmLmVtYj1ubi5FbWJlZGRpbmcoMjU2LGQpOyBzZWxmLnBvcz1Qb3MoZCkKICAgICAgICBsPW5uLlRyYW5zZm9ybWVyRW5jb2RlckxheWVyKGQsNCw0KmQsZHJvcG91dD0wLGJhdGNoX2ZpcnN0PVRydWUsbm9ybV9maXJzdD1UcnVlLGFjdGl2YXRpb249ImdlbHUiKQogICAgICAgIHNlbGYubmV0PW5uLlRyYW5zZm9ybWVyRW5jb2RlcihsLDIsZW5hYmxlX25lc3RlZF90ZW5zb3I9RmFsc2UpOyBzZWxmLm5vcm09bm4uTGF5ZXJOb3JtKGQpCiAgICBkZWYgZm9yd2FyZChzZWxmLHRvayk6CiAgICAgICAgeD1zZWxmLnBvcyhzZWxmLmVtYih0b2spKTsgVD10b2suc2l6ZSgxKTsgbT10b3JjaC50cml1KHRvcmNoLmZ1bGwoKFQsVCksZmxvYXQoIi1pbmYiKSxkZXZpY2U9dG9rLmRldmljZSxkdHlwZT14LmR0eXBlKSwxKQogICAgICAgIHg9c2VsZi5uZXQoeCxtYXNrPW0saXNfY2F1c2FsPVRydWUpOyByZXR1cm4gRi5saW5lYXIoc2VsZi5ub3JtKHgpLHNlbGYuZW1iLndlaWdodCkKCmRlZiBwYXJhbXMobSk6IHJldHVybiBzdW0ocC5udW1lbCgpIGZvciBwIGluIG0ucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkKYmlvPUZseUxNKGRzdCk7IHJuZD1GbHlMTShkc3RfcmFuZCkKdGFyZ2V0PXBhcmFtcyhiaW8pOyBjaG9pY2VzPVtdCmZvciBkIGluIHJhbmdlKDQ4LDE5MywxNik6CiAgICBpZiBkJTQ9PTA6CiAgICAgICAgcT1UckxNKGQpOyBjaG9pY2VzLmFwcGVuZCgoYWJzKHBhcmFtcyhxKS10YXJnZXQpLGQscGFyYW1zKHEpKSkKXyxkLF89bWluKGNob2ljZXMpOyB0cmFucz1UckxNKGQpCnByaW50KCJwYXJhbXMiLHsidHJhbnNmb3JtZXIiOnBhcmFtcyh0cmFucyksImZseV9iaW8iOnBhcmFtcyhiaW8pLCJmbHlfcmV3aXJlZCI6cGFyYW1zKHJuZCl9KQoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgZXZhbHVhdGUobSk6CiAgICBtLmV2YWwoKTsgbHM9W10KICAgIGZvciBzdCBpbiB2c3RhcnRzOgogICAgICAgIHgseT1iYXRjaCh2YSxzdCk7IGxzLmFwcGVuZChGLmNyb3NzX2VudHJvcHkobSh4KS5yZXNoYXBlKC0xLDI1NikseS5yZXNoYXBlKC0xKSkuaXRlbSgpKQogICAgY2U9ZmxvYXQobnAubWVhbihscykpOyByZXR1cm4geyJjZSI6Y2UsImJwYiI6Y2UvbWF0aC5sb2coMiksInBwbF9ieXRlIjptYXRoLmV4cChjZSl9CmRlZiB0cmFpbihtLG5hbWUpOgogICAgdG9yY2gubWFudWFsX3NlZWQoU0VFRCk7IG09bS50byhkZXZpY2UpOyBvcHQ9dG9yY2gub3B0aW0uQWRhbVcobS5wYXJhbWV0ZXJzKCksbHI9M2UtNCx3ZWlnaHRfZGVjYXk9LjAxKQogICAgc2NhbGVyPXRvcmNoLmN1ZGEuYW1wLkdyYWRTY2FsZXIoZW5hYmxlZD1kZXZpY2UudHlwZT09ImN1ZGEiKTsgdDA9dGltZS5wZXJmX2NvdW50ZXIoKTsgc2Vlbj0wCiAgICBmb3IgaSBpbiByYW5nZShTVEVQUyk6CiAgICAgICAgeCx5PWJhdGNoKHRyLHN0YXJ0c1tpXSk7IG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KGRldmljZV90eXBlPWRldmljZS50eXBlLGR0eXBlPXRvcmNoLmZsb2F0MTYsZW5hYmxlZD1kZXZpY2UudHlwZT09ImN1ZGEiKToKICAgICAgICAgICAgbG9zcz1GLmNyb3NzX2VudHJvcHkobSh4KS5yZXNoYXBlKC0xLDI1NikseS5yZXNoYXBlKC0xKSkKICAgICAgICBzY2FsZXIuc2NhbGUobG9zcykuYmFja3dhcmQoKTsgc2NhbGVyLnVuc2NhbGVfKG9wdCk7IHRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtLnBhcmFtZXRlcnMoKSwxLjApOyBzY2FsZXIuc3RlcChvcHQpOyBzY2FsZXIudXBkYXRlKCk7IHNlZW4rPXgubnVtZWwoKQogICAgICAgIGlmIGklMjA9PTAgb3IgaT09U1RFUFMtMTogcHJpbnQobmFtZSxpLGZsb2F0KGxvc3MpKQogICAgZHQ9dGltZS5wZXJmX2NvdW50ZXIoKS10MDsgcj1ldmFsdWF0ZShtKTsgci51cGRhdGUoeyJwYXJhbXMiOnBhcmFtcyhtKSwidHJhaW5fYnl0ZXNfcyI6c2Vlbi9kdH0pOyByZXR1cm4gbSxyCgptb2RlbHM9e307IHJlc3VsdHM9e30KZm9yIG5hbWUsbSBpbiBbKCJUcmFuc2Zvcm1lciIsdHJhbnMpLCgiRmx5Q2VOTiBiaW9sb2dpY2FsIixiaW8pLCgiRmx5Q2VOTiByZXdpcmVkIixybmQpXToKICAgIG1vZGVsc1tuYW1lXSxyZXN1bHRzW25hbWVdPXRyYWluKG0sbmFtZSkKCmRlZiBiZW5jaChtLEw9U0VRLHJlcHM9MTApOgogICAgbS5ldmFsKCk7IHg9dG9yY2gucmFuZGludCgwLDI1NiwoOCxMKSxkZXZpY2U9ZGV2aWNlKQogICAgaWYgZGV2aWNlLnR5cGU9PSJjdWRhIjogdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpOyB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciBfIGluIHJhbmdlKDIpOiBtKHgpCiAgICAgICAgaWYgZGV2aWNlLnR5cGU9PSJjdWRhIjogdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpOyB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKICAgICAgICB0PXRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBmb3IgXyBpbiByYW5nZShyZXBzKTogbSh4KQogICAgICAgIGlmIGRldmljZS50eXBlPT0iY3VkYSI6IHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgZHQ9dGltZS5wZXJmX2NvdW50ZXIoKS10CiAgICByZXR1cm4gKDgqTCpyZXBzL2R0LCB0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkvMioqMjAgaWYgZGV2aWNlLnR5cGU9PSJjdWRhIiBlbHNlIGZsb2F0KCJuYW4iKSkKZm9yIG5hbWUsbSBpbiBtb2RlbHMuaXRlbXMoKToKICAgIHMsbWVtPWJlbmNoKG0pOyByZXN1bHRzW25hbWVdLnVwZGF0ZSh7ImZvcndhcmRfYnl0ZXNfcyI6cywicGVha192cmFtX21iIjptZW19KQoKc3VtbWFyeT1wZC5EYXRhRnJhbWUocmVzdWx0cykuVApkaXNwbGF5KHN1bW1hcnkuc29ydF92YWx1ZXMoImJwYiIpKQpiaW9fcj1yZXN1bHRzWyJGbHlDZU5OIGJpb2xvZ2ljYWwiXTsgcm5kX3I9cmVzdWx0c1siRmx5Q2VOTiByZXdpcmVkIl07IHRyX3I9cmVzdWx0c1siVHJhbnNmb3JtZXIiXQpyZXBvcnQ9ewogInRvcG9sb2d5X2dhaW5fcGN0X2JwYiI6MTAwKihybmRfclsiYnBiIl0tYmlvX3JbImJwYiJdKS9ybmRfclsiYnBiIl0sCiAiZmx5X3ZzX3RyYW5zZm9ybWVyX2JwYl9nYXBfcGN0IjoxMDAqKGJpb19yWyJicGIiXS10cl9yWyJicGIiXSkvdHJfclsiYnBiIl0sCiAiZmx5X3ZzX3RyYW5zZm9ybWVyX3NwZWVkX3JhdGlvIjpiaW9fclsiZm9yd2FyZF9ieXRlc19zIl0vdHJfclsiZm9yd2FyZF9ieXRlc19zIl0sCiAiZmx5X3ZzX3RyYW5zZm9ybWVyX3ZyYW1fcmF0aW8iOmJpb19yWyJwZWFrX3ZyYW1fbWIiXS90cl9yWyJwZWFrX3ZyYW1fbWIiXSBpZiB0cl9yWyJwZWFrX3ZyYW1fbWIiXT4wIGVsc2UgTm9uZQp9CnByaW50KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yKSkKCm91dD1wYXRobGliLlBhdGgoIi9jb250ZW50L1RpbnlDZU5OLUxNL3Jlc3VsdHMvZmx5Y2Vubl90cmFuc2Zvcm1lcl9wcm9iZSIpOyBvdXQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCnN1bW1hcnkudG9fY3N2KG91dC8ic3VtbWFyeS5jc3YiKTsgKG91dC8icmVwb3J0Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyJjb25maWciOnsibm9kZXMiOk5PREVTLCJlZGdlcyI6TUFYX0VER0VTLCJzZXEiOlNFUSwiYmF0Y2giOkJBVENILCJzdGVwcyI6U1RFUFN9LCJyZXN1bHRzIjpyZXN1bHRzLCJyZXBvcnQiOnJlcG9ydH0saW5kZW50PTIpKQpwcmludCgic2F2ZWQ6IixvdXQpCg==').decode())
